# Data Preprocessing

Use this notebook to clean raw measurements, encode features, and prepare model-ready tables for sklearn pipelines. It also creates molecule-aware splits for each metal so every molecule is represented in training first, then testing, and then at least one calibration split when enough rows are available.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/41598_2020_71255_MOESM1_ESM.xlsx")
OUTPUT_ROOT = Path("../data/splits")
TARGET_COL = "Log K1"
METAL_COL = "Metal"
MOLECULE_COL = "Molecule"
GROUP_COLS = [MOLECULE_COL, "Isomer"]
RANDOM_SEED = 42


def load_data() -> pd.DataFrame:
    df = pd.read_excel(DATA_PATH, sheet_name="lanthanides_final_dataset_for_M")
    df.columns = [column.strip() for column in df.columns]
    df[METAL_COL] = df[METAL_COL].astype(str).str.strip()
    df[MOLECULE_COL] = df[MOLECULE_COL].astype(str).str.strip()
    df["Isomer"] = df["Isomer"].astype(str).str.strip()
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    return df


def sanitize_label(label: str) -> str:
    return (
        label.strip()
        .replace("+++", "_3plus")
        .replace("++", "_2plus")
        .replace("+", "plus")
        .replace("/", "_")
        .replace(" ", "_")
    )


def target_split_rows(total_rows: int, target_fractions: dict[str, float]) -> dict[str, float]:
    return {name: total_rows * fraction for name, fraction in target_fractions.items()}


def choose_deficit_split(assigned_rows: dict[str, int], target_rows: dict[str, float]) -> str:
    return min(
        target_rows,
        key=lambda name: (
            (assigned_rows[name] / target_rows[name]) if target_rows[name] > 0 else np.inf,
            assigned_rows[name],
            name,
        ),
    )


def assign_rows_by_molecule(
    frame: pd.DataFrame,
    target_fractions: dict[str, float],
    molecule_col: str,
    random_seed: int,
) -> pd.DataFrame:
    working = frame.reset_index(names="source_index").copy()
    working["split"] = None

    split_priority = ["train", "test", "calibration_global", "calibration_local"]
    target_rows = target_split_rows(len(working), target_fractions)
    assigned_rows = {name: 0 for name in target_fractions}

    for molecule_index, (_, molecule_rows) in enumerate(working.groupby(molecule_col, sort=False)):
        molecule_rows = molecule_rows.sample(frac=1, random_state=random_seed + molecule_index).reset_index(drop=True)
        row_indices = molecule_rows["source_index"].tolist()
        n_rows = len(row_indices)

        mandatory_splits = split_priority[: min(n_rows, len(split_priority))]
        for row_index, split_name in zip(row_indices[: len(mandatory_splits)], mandatory_splits):
            working.loc[working["source_index"] == row_index, "split"] = split_name
            assigned_rows[split_name] += 1

        for row_index in row_indices[len(mandatory_splits):]:
            split_name = choose_deficit_split(assigned_rows, target_rows)
            working.loc[working["source_index"] == row_index, "split"] = split_name
            assigned_rows[split_name] += 1

    return working


def materialize_splits(
    frame: pd.DataFrame,
    metal: str,
    target_fractions: dict[str, float],
    molecule_col: str,
    random_seed: int,
) -> dict[str, pd.DataFrame]:
    metal_frame = frame.loc[frame[METAL_COL] == metal].copy()
    assigned = assign_rows_by_molecule(
        frame=metal_frame,
        target_fractions=target_fractions,
        molecule_col=molecule_col,
        random_seed=random_seed,
    )

    splits = {
        split_name: assigned.loc[assigned["split"] == split_name].drop(columns=["split"]).copy()
        for split_name in target_fractions
    }
    splits["_manifest"] = assigned[["source_index", MOLECULE_COL, "Isomer", METAL_COL, "split"]].copy()
    return splits


def save_splits(splits: dict[str, pd.DataFrame], metal: str) -> Path:
    metal_dir = OUTPUT_ROOT / sanitize_label(metal)
    metal_dir.mkdir(parents=True, exist_ok=True)

    for split_name, split_frame in splits.items():
        if split_name == "_manifest":
            split_frame.to_csv(metal_dir / "split_manifest.csv", index=False)
        else:
            split_frame.to_csv(metal_dir / f"{split_name}.csv", index=False)

    return metal_dir


# Fractions are expressed over seven equal parts: 4/7 train, 1/7 calibration-global,
# 1/7 calibration-local, and 1/7 test.
target_fractions = {
    "train": 4 / 7,
    "calibration_global": 1 / 7,
    "calibration_local": 1 / 7,
    "test": 1 / 7,
}


metals_to_split = ["Nd+++"]
df = load_data()

for metal_of_interest in metals_to_split:
    splits = materialize_splits(
        frame=df,
        metal=metal_of_interest,
        target_fractions=target_fractions,
        molecule_col=MOLECULE_COL,
        random_seed=RANDOM_SEED,
    )
    output_dir = save_splits(splits, metal_of_interest)

    print(f"Saved {metal_of_interest} splits to {output_dir}")
    for split_name in ["train", "calibration_global", "calibration_local", "test"]:
        split_frame = splits[split_name]
        print(
            f"{split_name}: {len(split_frame)} rows, "
            f"{split_frame[MOLECULE_COL].nunique()} unique molecules, "
            f"{split_frame['Isomer'].nunique()} unique isomers"
        )


Saved Nd+++ splits to ../data/splits/Nd_3plus
train: 418 rows, 414 unique molecules, 414 unique isomers
calibration_global: 87 rows, 58 unique molecules, 58 unique isomers
calibration_local: 87 rows, 37 unique molecules, 37 unique isomers
test: 109 rows, 109 unique molecules, 109 unique isomers
